# EDA-basierte Random-Forest-Baseline

**Autor:** duac1011 · **Datum:** 2026-10-02

**Ziel:** Diese eigene Baseline prüft die konkreten EDA-Empfehlungen als reproduzierbare Pipeline-Varianten. Sie trennt Spektralinformation von den potenziell fragilen Kontextmerkmalen `AEZ` und `Month` und untersucht, ob Vegetationsindizes über die Rohbänder hinaus helfen.

**Hypothese:** Ein Random Forest mit bereinigten Spektralbändern kann die bestehende Random-Forest-Baseline erreichen, ohne sich auf Orts- und Kalendereffekte zu stützen.

**Regel:** Alle Varianten werden ausschließlich per Cross-Validation auf dem gemeinsamen Trainingsanteil verglichen. Nur der CV-Gewinner wird genau einmal auf dem Holdout bewertet.

## 1. Bibliotheken und EDA-Konfiguration

Die vorhandene Random-Forest-Baseline ist im gemeinsamen Experiment-Log dokumentiert, aber ihre MLflow-Läufe sind lokal bei der ausführenden Person. Dieses Notebook lädt deshalb keine fremden MLflow-Runs. Der Teamvergleich erfolgt über `docs/modelle/experimente.md`.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier

from awp2.config import MODEL_DOCS_FIGURES_DIR, SEED
from awp2.data import load_split, valid_combinations
from awp2.evaluation import plot_confusion_matrices
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.plots import (
    plot_class_recall,
    plot_top_confusions,
    plot_tuning_candidates,
    save_doc_figure,
)
from awp2.preprocessing import PreprocessingConfig

## 2. EDA-konforme Daten- und Feature-Pipelines

Jede Variante verwendet die gemeinsame Vorverarbeitung: vollständig leere Bänder werden auf dem jeweiligen Trainingsanteil entfernt und einzelne Lücken entlang der Wellenlänge interpoliert. Der Random Forest benötigt keine Skalierung. Die Indizes werden nach der Interpolation innerhalb derselben Pipeline ergänzt.

## 3. Referenzlauf: Spektralbänder ohne Metadaten

Diese Referenz verwendet nur Rohbänder. `AEZ` und `Month` bleiben bewusst außen vor, weil die EDA ihre starke Labelassoziation als möglichen Shortcut identifiziert hat.

## 4. Kontextlauf: Spektralbänder mit AEZ und Monat

Diese Variante prüft, ob der Kontext im gemeinsamen Split nützt. Sie ist nicht automatisch robuster für einen späteren Einsatz.

## 5. Domänenlauf: Rohbänder mit Vegetationsindizes

NDVI, NDRE, PRI und NDWI werden zusätzlich zu den Rohbändern berechnet. Die EDA hat Gruppenunterschiede gezeigt; erst die CV entscheidet, ob sie zusätzliche Vorhersageinformation liefern.

In [ ]:
RF_GRID = {
    "max_depth": [None, 20],
    "min_samples_leaf": [1, 3],
    "max_features": ["sqrt", 0.3],
}

variants = {
    "Spektren": (
        "spectral",
        PreprocessingConfig(use_meta=False, scale=False, use_vegetation_indices=False),
    ),
    "Spektren + Kontext": (
        "context",
        PreprocessingConfig(use_meta=True, scale=False, use_vegetation_indices=False),
    ),
    "Spektren + Indizes": (
        "indices",
        PreprocessingConfig(use_meta=False, scale=False, use_vegetation_indices=True),
    ),
}

variant_tuning = {}
for label, (slug, preprocessing) in variants.items():
    variant_tuning[label] = tune(
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        ),
        TuneConfig(
            name=f"rf_{slug}_search",
            approach="baseline",
            description=f"EDA-Baseline, Variante: {label}.",
            param_grid=RF_GRID,
            preprocessing=preprocessing,
        ),
    )

cv_comparison = pd.DataFrame(
    {
        "Variante": list(variant_tuning),
        "CV BAcc kombiniert": [result.best_score for result in variant_tuning.values()],
        "Beste Parameter": [result.best_params for result in variant_tuning.values()],
        "MLflow-Tuning-Run": [result.run_id for result in variant_tuning.values()],
    }
).sort_values("CV BAcc kombiniert", ascending=False)
display(cv_comparison)

best_variant = cv_comparison.iloc[0]["Variante"]
tuned = variant_tuning[best_variant]
selected_preprocessing = variants[best_variant][1]
print(f"CV-Gewinner: {best_variant} ({tuned.best_score:.3f})")

## 6. Besten EDA-Baseline-Kandidaten per Cross-Validation abstimmen

Die Tabelle zeigt die Auswahlentscheidung. Die gemeinsame primäre Kennzahl ist die kombinierte Balanced Accuracy. Der Holdout bleibt bis zur folgenden Zelle unangetastet.

## 7. Bestes Modell einmal auf dem Holdout bewerten

Nur der oben per CV ausgewählte Kandidat wird auf dem gemeinsamen Validierungsanteil trainiert und einmal bewertet. Ein schlechter oder guter Holdout-Score führt nicht zu weiteren Änderungen in diesem Notebook.

In [ ]:
eda_rf = run(
    tuned.best_model,
    RunConfig(
        name="rf_eda_baseline",
        approach="baseline",
        description=f"EDA-basierter Random Forest, CV-Gewinner: {best_variant}.",
        preprocessing=selected_preprocessing,
        tuning_run=tuned.run_id,
    ),
)

display(pd.Series(eda_rf.metrics.model_dump(), name="Holdout-Metrik"))
print(f"MLflow-Run: {eda_rf.run_id}")

## 8. Tuning- und Holdout-Metriken einordnen

Die CV-Tabelle vergleicht die drei EDA-Varianten. Der Tuning-Plot zeigt für die ausgewählte Variante, ob ihr einmaliger Holdout-Score im Bereich der CV-Streuung liegt. Der Vergleich mit anderen Teamläufen erfolgt im gemeinsamen Experiment-Log, nicht über lokale MLflow-Datenbanken.

In [ ]:
ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="max_features",
    validation_score=eda_rf.metrics.bacc_combined,
)
ax.set_title(f"Tuning EDA-Baseline: {best_variant}")
save_doc_figure(ax.figure, "spectral_baseline_tuning", MODEL_DOCS_FIGURES_DIR)

## 9. Klassenfehler, ungültige Crop-Stage-Paare und Metadatenabhängigkeit

Neben den Gesamtmetriken zeigen Confusion Matrices und Recall, welche Klassen weiterhin problematisch sind. Die Fehlerquote nach AEZ und Monat ist reine Diagnose der finalen Vorhersagen; sie steuert keine weitere Modellauswahl.

In [ ]:
fig = plot_confusion_matrices(eda_rf.y_val, eda_rf.y_pred)
save_doc_figure(fig, "spectral_baseline_confusion", MODEL_DOCS_FIGURES_DIR)

fig = plot_top_confusions(eda_rf.y_val, eda_rf.y_pred)
save_doc_figure(fig, "spectral_baseline_verwechslungen", MODEL_DOCS_FIGURES_DIR)

fig = plot_class_recall(eda_rf.y_val, eda_rf.y_pred)
save_doc_figure(fig, "spectral_baseline_recall", MODEL_DOCS_FIGURES_DIR)

split = load_split()
valid_pairs = {(pair.crop, pair.stage) for pair in valid_combinations(split.y_train)}
predicted_pairs = list(eda_rf.y_pred[["Crop", "Stage"]].itertuples(index=False, name=None))
invalid_pairs = pd.Series(
    [pair for pair in predicted_pairs if pair not in valid_pairs],
    dtype="object",
).value_counts()
print(f"Anteil ungültiger Paare: {eda_rf.metrics.invalid_combinations:.1%}")
display(invalid_pairs.rename("Anzahl").to_frame())

validation_metadata = split.X_val[["AEZ", "Month"]].copy()
validation_metadata["Crop korrekt"] = eda_rf.y_val["Crop"].eq(eda_rf.y_pred["Crop"]).to_numpy()
validation_metadata["Stage korrekt"] = eda_rf.y_val["Stage"].eq(eda_rf.y_pred["Stage"]).to_numpy()
metadata_error_rates = validation_metadata.groupby(["AEZ", "Month"], observed=True).agg(
    **{
        "Zeilen": ("AEZ", "size"),
        "Crop-Fehleranteil": ("Crop korrekt", lambda values: 1 - values.mean()),
        "Stage-Fehleranteil": ("Stage korrekt", lambda values: 1 - values.mean()),
    }
)
display(metadata_error_rates.sort_values("Stage-Fehleranteil", ascending=False))

## 10. Ergebnisse und nächste Modellvarianten dokumentieren

### Ergebnis dieser EDA-Merkmalsablage

| Pipeline-Variante | CV BAcc kombiniert | Entscheidung |
| --- | --- | --- |
| Rohspektren | 0,570 | deutlich unter der Referenz |
| Rohspektren + NDVI, NDRE, PRI, NDWI | 0,590 | kein messbarer Zusatznutzen gegenüber den Rohbändern |
| Rohspektren + AEZ, Month | **0,750** | per CV ausgewählt |

Der einmalige Holdout-Lauf der Kontextvariante erreicht BAcc 0,890 für `Crop`, 0,870 für `Stage` und 0,743 kombiniert. Der Wert liegt im Bereich der CV und reproduziert damit die bereits dokumentierte Random-Forest-Baseline, statt sie zu verbessern.

`AEZ` und `Month` enthalten in diesem Split sehr starke Information. Das bestätigt den EDA-Befund zu möglichen Shortcut-Merkmalen, belegt aber nicht, dass das Modell bei einer räumlich oder zeitlich anders verteilten Anwendung genauso robust ist. Die Vegetationsindizes liefern neben den Rohbändern in dieser Konfiguration keinen Zusatznutzen.

Der Random Forest sagt 0,3 % ungültige Crop-Stage-Paare voraus, hier fünfmal `winter_wheat|Early_Mid`. Er bleibt deshalb eine Diagnose- und Referenzvariante, keine abgabefertige Vorhersage. Das Erzwingen gültiger Paare ist als eigene, erneut per CV zu prüfende Nachverarbeitungs- oder kombinierte Modellvariante umzusetzen.

Die tatsächlichen Kennzahlen und Run-IDs stehen im gemeinsamen Experiment-Log und auf der Baseline-Seite. Nächste getrennt zu prüfende Ideen sind PCA gegen Rohbänder, zyklische Monatskodierung sowie ein kombinierter oder hierarchischer Crop-Stage-Ansatz. Jede Idee beginnt erneut mit CV auf dem Trainingsanteil und erhält erst danach genau eine Holdout-Bewertung.